<a href="https://colab.research.google.com/github/14marcos1/curso_colab_2026/blob/main/acre_GEMINI_2010_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GEMINI GERAL


 Gerar código


Atue como um especialista em engenharia de dados de saúde pública e crie um script em Python para o Google Colab para extrair dados do Sistema de Informações sobre Mortalidade (SIM/DATASUS).


Siga RIGOROSAMENTE os parâmetros e as regras de arquitetura fornecidas.


---

PARÂMETROS DA PESQUISA (Ajuste se necessário):

- Estado (UF): [AC]

- Período: [2010 a 2023]

- Grupo 1: "Neoplasias" -> CIDs-10 iniciados com as letras ['C', 'D']

- Grupo 2: "Aparelho Circulatório" -> CIDs-10 iniciados com a letra ['I']

---


REGRAS OBRIGATÓRIAS DE ARQUITETURA DO CÓDIGO (NÃO MUDAR OU OMITIR):


1. INSTALAÇÃO DE DEPENDÊNCIAS DE SISTEMA (LINUX E PYTHON):

   Instale no início do script os pacotes de sistema necessários para converter os arquivos do DATASUS:

   !apt-get update -qq && !apt-get install -y -qq p7zip-full

   !pip install pyreaddbc dbfread pandas -q


2. IMPORTAÇÃO E SINTAXE OBRIGATÓRIA DO DBC2DBF (CRÍTICO):

   - Importe a função usando exatamente: `from pyreaddbc import dbc2dbf`

   - Para converter os arquivos, passe estritamente DUAS strings com os caminhos completos de entrada e saída: `dbc2dbf(caminho_dbc, caminho_dbf)`

   - Nunca use `dbc2dbf.dbc2dbf()` nem métodos sem o parâmetro do arquivo de saída.


3. BAIXAMENTO E CONVERSÃO (NÃO USE PYSUS ONLINE):

   - Baixe os arquivos do FTP oficial: "ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ANO}.DBC"

   - Adicione tentativa de download com extensão minúscula (.dbc) caso a maiúscula falhe.

   - Leia o arquivo .dbf convertido com `dbfread.DBF(..., encoding="latin1")`.

   - Remova os arquivos temporários (.dbc e .dbf) do disco a cada loop para não lotar a memória do Colab.


4. TRATAMENTO DA COLUNA E AGRUPAMENTO:

   - Filtre a coluna 'CAUSABAS' utilizando os prefixos de CID fornecidos nos parâmetros.

   - Conte o total de óbitos ano a ano para cada grupo ("Neoplasias" e "Aparelho Circulatório").


5. PADRONIZAÇÃO RÍGIDA DA SAÍDA:

   - O arquivo de saída DEVE ser salvo OBRIGATORIAMENTE no formato: f"mortalidade_{UF.lower()}_consolidado.csv" (Exemplo para o Acre: "mortalidade_ac_consolidado.csv").

   - A planilha DEVE conter exatamente 3 colunas com estes nomes em maiúsculo: ANO, GRUPO, OBITOS.

   - Inclua no final o comando `google.colab.files.download()` para baixar a planilha no computador.

   - Entregue o código completo em um ÚNICO bloco executável.

In [2]:
# Instalação das dependências de sistema e pacotes Python necessários
!apt-get update -qq && !apt-get install -y -qq p7zip-full
!pip install pyreaddbc dbfread pandas -q

import os
import urllib.request
import pandas as pd
from pyreaddbc import dbc2dbf
import dbfread
from google.colab import files

# Parâmetros da pesquisa
UF = "AC"
ANOS = range(2010, 2024)

dados_consolidados = []

for ano in ANOS:
    ano_2dig = str(ano)[2:]
    ano_4dig = str(ano)

    # Tentativas de URLs no FTP oficial do DATASUS (com variações de maiúsculas/minúsculas)
    urls_tentativas = [
        f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ano_2dig}.DBC",
        f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ano_2dig}.dbc",
        f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ano_4dig}.DBC",
        f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DO{UF}{ano_4dig}.dbc",
        f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/do{UF.lower()}{ano_2dig}.dbc",
        f"ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/do{UF.lower()}{ano_4dig}.dbc"
    ]

    arquivo_baixado = False
    arquivo_dbc = f"DO{UF}{ano}.DBC"
    arquivo_dbf = f"DO{UF}{ano}.dbf"

    for url in urls_tentativas:
        try:
            urllib.request.urlretrieve(url, arquivo_dbc)
            arquivo_baixado = True
            break
        except Exception:
            continue

    if not arquivo_baixado:
        print(f"Aviso: Arquivo para o ano {ano} não encontrado no FTP.")
        continue

    try:
        # Conversão DBC para DBF usando exatamente a sintaxe exigida
        dbc2dbf(arquivo_dbc, arquivo_dbf)

        # Leitura do DBF com encoding latin1
        table = dbfread.DBF(arquivo_dbf, encoding="latin1")
        df_ano = pd.DataFrame(iter(table))

        # Padronizar nomes das colunas para maiúsculas
        df_ano.columns = [col.upper() for col in df_ano.columns]

        if 'CAUSABAS' in df_ano.columns:
            df_ano['CAUSABAS'] = df_ano['CAUSABAS'].fillna('').astype(str).str.strip().str.upper()

            # Filtros por Grupo de CIDs-10
            # Grupo 1: Neoplasias (iniciados com C ou D)
            cond_neoplasias = df_ano['CAUSABAS'].str.startswith(('C', 'D'))

            # Grupo 2: Aparelho Circulatório (iniciados com I)
            cond_circulatorio = df_ano['CAUSABAS'].str.startswith('I')

            obitos_neoplasias = df_ano[cond_neoplasias].shape[0]
            obitos_circulatorio = df_ano[cond_circulatorio].shape[0]

            dados_consolidados.append({
                'ANO': ano,
                'GRUPO': 'Neoplasias',
                'OBITOS': obitos_neoplasias
            })
            dados_consolidados.append({
                'ANO': ano,
                'GRUPO': 'Aparelho Circulatório',
                'OBITOS': obitos_circulatorio
            })

    except Exception as e:
        print(f"Erro ao processar o arquivo do ano {ano}: {e}")

    finally:
        # Remoção dos arquivos temporários a cada loop
        if os.path.exists(arquivo_dbc):
            os.remove(arquivo_dbc)
        if os.path.exists(arquivo_dbf):
            os.remove(arquivo_dbf)

# Criação do DataFrame final e salvamento no formato solicitado
df_resultado = pd.DataFrame(dados_consolidados)
nome_arquivo_saida = f"mortalidade_{UF.lower()}_consolidado.csv"

df_resultado.to_csv(nome_arquivo_saida, index=False, encoding='utf-8-sig')
print(f"\nProcessamento concluído. Arquivo gerado: {nome_arquivo_saida}")

display(df_resultado)

# Download automático do arquivo gerado no Google Colab
files.download(nome_arquivo_saida)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/bin/bash: line 1: !apt-get: command not found
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.1 MB/s eta 0:00:00

Processamento concluído. Arquivo gerado: mortalidade_ac_consolidado.csv


,ANO,GRUPO,OBITOS
0,2010,Neoplasias,353
1,2010,Aparelho Circulatório,606
2,2011,Neoplasias,399
3,2011,Aparelho Circulatório,715
4,2012,Neoplasias,497
5,2012,Aparelho Circulatório,659
6,2013,Neoplasias,498
7,2013,Aparelho Circulatório,698
8,2014,Neoplasias,523
9,2014,Aparelho Circulatório,762


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

grafico

Atue como um especialista em visualização de dados de saúde pública e crie um script em Python para o Google Colab para gerar um gráfico de linhas interativo e profissional a partir do arquivo CSV gerado na etapa anterior.
Siga RIGOROSAMENTE os parâmetros e as regras de arquitetura fornecidas.
---
REGRAS OBRIGATÓRIAS DE ARQUITETURA DO CÓDIGO (NÃO MUDAR OU OMITIR):
1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS:
   - Instale e importe as bibliotecas necessárias para plotagem interativa e manipulação de dados:
     !pip install plotly pandas -q
   - Importe pandas as pd e plotly.express as px (ou plotly.graph_objects as go).
2. LEITURA FLEXÍVEL DO ARQUIVO CSV (CRÍTICO - ANTI-FALHA):
   - O script deve procurar o arquivo na pasta atual do Colab.
   - Faça uma busca dinâmica testando os arquivos que terminam com "_consolidado.csv" ou o padrão "mortalidade_ac_consolidado.csv".
   - Adicione uma checagem de segurança: se nenhum arquivo CSV for encontrado, mostre uma mensagem explicativa orientando o usuário a fazer o upload do arquivo CSV gerado na Etapa 1.
3. TRATAMENTO E PADRONIZAÇÃO DE DADOS:
   - Garanta que as colunas 'ANO' e 'OBITOS' sejam convertidas para tipo numérico (pd.to_numeric).
   - Garanta que os valores do eixe X (anos) sejam exibidos como NÚMEROS INTEIROS (ex: 2010, 2011, 2012) e NUNCA como decimais (ex: 2010.0 ou 2010.5).
4. DESIGN E ESTILIZAÇÃO DO GRÁFICO (QUALIDADE DE PUBLICAÇÃO):
   - Crie um gráfico de linhas mostrando a evolução temporal dos óbitos por GRUPO ("Neoplasias" vs "Aparelho Circulatório").
   - Destaque as linhas com marcadores de pontos em cada ano (markers=True).
   - Adicione um título claro informando o Estado e o Período analisado.
   - Configure o eixe X para mostrar todos os anos explicitamente (marcações ano a ano).
   - Use um tema visual limpo (plotly_white ou similar) com legenda visível e rótulos de eixos bem definidos (Ex: Eixo X: "Ano", Eixo Y: "Número de Óbitos").
5. EXIBIÇÃO E DOWNLOAD:
   - Exiba o gráfico interativo diretamente na tela do notebook Colab.
   - Salve o gráfico em formato HTML interativo (mortalidade_grafico_interativo.html) e adicione o comando google.colab.files.download() para baixar o arquivo no computador.
   - Entregue o código completo em um ÚNICO bloco executável.

In [4]:
import glob
import os
import pandas as pd
import plotly.express as px

# 1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# Executa a instalação silenciosa das dependências necessárias para o ambiente Colab
get_ipython().system("pip install plotly pandas -q")

# 2. LEITURA FLEXÍVEL DO ARQUIVO CSV
arquivos_encontrados = glob.glob("*_consolidado.csv")

if not arquivos_encontrados and os.path.exists(
    "mortalidade_ac_consolidado.csv"
):
  arquivos_encontrados = ["mortalidade_ac_consolidado.csv"]

if not arquivos_encontrados:
  print(
      "❌ ATENÇÃO: Nenhum arquivo CSV consolidado encontrado na pasta atual.\nPor"
      " favor, faça o upload do arquivo CSV gerado na Etapa 1 (ex:"
      " 'mortalidade_ac_consolidado.csv') para prosseguir."
  )
else:
  arquivo_csv = arquivos_encontrados[0]
  print(f"📂 Arquivo carregado com sucesso: {arquivo_csv}")

  # Carregamento dos dados
  df = pd.read_csv(arquivo_csv)

  # 3. TRATAMENTO E PADRONIZAÇÃO DE DADOS
  df["ANO"] = pd.to_numeric(df["ANO"], errors="coerce")
  df["OBITOS"] = pd.to_numeric(df["OBITOS"], errors="coerce")

  # Limpeza de valores nulos essenciais
  df = df.dropna(subset=["ANO", "OBITOS", "GRUPO"])

  # Conversão estrita do eixo X para números inteiros
  df["ANO"] = df["ANO"].astype(int)

  # Ordenação cronológica
  df = df.sort_values("ANO")

  # 4. DESIGN E ESTILIZAÇÃO DO GRÁFICO (QUALIDADE DE PUBLICAÇÃO)
  grupos_desejados = ["Neoplasias", "Aparelho Circulatório"]
  df_filtrado = df[df["GRUPO"].isin(grupos_desejados)]

  if df_filtrado.empty:
    df_filtrado = df  # Fallback caso os nomes exatos difiram na base original

  # Obtenção de todos os anos únicos para marcação explícita no Eixo X
  anos_unicos = sorted(df_filtrado["ANO"].unique())

  # Construção do gráfico de linhas interativo
  fig = px.line(
      df_filtrado,
      x="ANO",
      y="OBITOS",
      color="GRUPO",
      markers=True,
      template="plotly_white",
      labels={
          "ANO": "Ano",
          "OBITOS": "Número de Óbitos",
          "GRUPO": "Grupo de Causas",
      },
  )

  # Ajustes finos de layout orientados a publicações científicas/técnicas
  fig.update_layout(
      title={
          "text": (
              "<b>Evolução Temporal da Mortalidade - GEMINI</b><br><sup>Comparativo"
              " entre Neoplasias e Aparelho Circulatório</sup>"
          ),
          "y": 0.92,
          "x": 0.5,
          "xanchor": "center",
          "yanchor": "top",
          "font": dict(size=16, color="#2c3e50"),
      },
      xaxis=dict(
          title="Ano",
          tickmode="array",
          tickvals=anos_unicos,
          ticktext=[str(int(ano)) for ano in anos_unicos],
          showgrid=True,
          gridcolor="#f0f0f0",
      ),
      yaxis=dict(
          title="Número de Óbitos", showgrid=True, gridcolor="#e5e5e5"
      ),
      legend=dict(
          title="<b>Grupo de Causas</b>",
          orientation="h",
          yanchor="bottom",
          y=1.05,
          xanchor="right",
          x=1,
      ),
      font=dict(family="Arial, sans-serif", size=12, color="#333333"),
      margin=dict(t=110, b=50, l=70, r=40),
      hovermode="x unified",
  )

  # 5. EXIBIÇÃO E DOWNLOAD
  fig.show()

  nome_arquivo_html = "mortalidade_grafico_interativo.html"
  fig.write_html(nome_arquivo_html)
  print(f"💾 Gráfico interativo exportado com sucesso: '{nome_arquivo_html}'")

  try:
    from google.colab import files

    files.download(nome_arquivo_html)
  except ImportError:
    print(
        "ℹ️ Executando fora do Google Colab. O arquivo HTML foi salvo na pasta"
        " local."
    )

📂 Arquivo carregado com sucesso: mortalidade_ac_consolidado.csv


💾 Gráfico interativo exportado com sucesso: 'mortalidade_grafico_interativo.html'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>